# CLV 거래활동·거래가치 이중축 임베딩 1단계 screening

seed 42, validation-only로 `M1`, `dual_clv_fixed`, `dual_shuffled_gate`, `dual_base_only`만 실행합니다. test와 holdout은 생성하지 않습니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys

drive.mount('/content/drive')
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
REVIEWED_SHA = 'cd1bb457a3e78b420539979d41c5ba8f5cc2ac70'
REPO_DIR = Path('/content/clv-m2-lightgcn-runner-dual')
assert not REPO_DIR.exists(), '새 Colab 런타임에서 실행하세요.'
subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REVIEWED_SHA], check=True)
actual_sha = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
assert actual_sha == REVIEWED_SHA
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('검토된 소스:', actual_sha)


In [ ]:
import json, torch
from lightgcn_clv_dual import MODELS, configure_dual_run, preflight_summary, run_experiment

DATASET_PRESET = 'hm_w60'  # 'hm_w60' 또는 'dunnhumby'
assert DATASET_PRESET in {'hm_w60', 'dunnhumby'}
RESULT_ROOT = Path('/content/drive/MyDrive/논문/data')
if DATASET_PRESET == 'hm_w60':
    cfg = configure_dual_run(
        'hm', short_hm=True,
        out_dir=str(RESULT_ROOT / 'results_clv_dual_hm_w60'),
        m1_checkpoint_dir=str(RESULT_ROOT / 'results_v3_hm_w60'),
    )
else:
    cfg = configure_dual_run(
        'dunnhumby',
        out_dir=str(RESULT_ROOT / 'results_clv_dual_dunnhumby'),
        m1_checkpoint_dir=str(RESULT_ROOT / 'results_v3_dunnhumby'),
    )
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
assert tuple(MODELS) == ('m1', 'dual_clv_fixed', 'dual_shuffled_gate', 'dual_base_only')
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))


In [ ]:
result_df = run_experiment(cfg)


In [ ]:
from IPython.display import display
import pandas as pd

display(result_df.sort_values(['model_id', 'lambda']))
print('선택 lambda:', result_df.attrs['selected_lambda'])
print('최종 screening 판정:', result_df.attrs['screening_decision'])
delta_path = Path(result_df.attrs['result_paths']['delta_csv'])
print('M1 대비 paired delta:')
display(pd.read_csv(delta_path).sort_values(['model_id', 'lambda', 'metric']))
print('결과 파일:')
for label, path in result_df.attrs['result_paths'].items():
    print(f' - {label}: {path}')
